# 01 — EDA Olist (Frete)

**Responsável:** Caio · etapa 01 — Preparação e EDA

**Objetivo:** inspecionar os cinco CSVs brutos (tamanhos, tipos, estatísticas, distribuições, nulos, duplicatas, inconsistências, extremos e relações entre chaves).

**Restrição:** não limpar, não codificar e não criar atributos finais. Os arquivos em `data/raw/` não são alterados.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid")

RAW = Path("../data/raw")


## 1. Carga dos arquivos brutos

In [ ]:
orders = pd.read_csv(RAW / "olist_orders_dataset.csv")
items = pd.read_csv(RAW / "olist_order_items_dataset.csv")
products = pd.read_csv(RAW / "olist_products_dataset.csv")
customers = pd.read_csv(RAW / "olist_customers_dataset.csv")
sellers = pd.read_csv(RAW / "olist_sellers_dataset.csv")

tabelas = {
    "orders": orders,
    "order_items": items,
    "products": products,
    "customers": customers,
    "sellers": sellers,
}

resumo = pd.DataFrame(
    {
        "linhas": [df.shape[0] for df in tabelas.values()],
        "colunas": [df.shape[1] for df in tabelas.values()],
        "memoria_mb": [round(df.memory_usage(deep=True).sum() / 1e6, 2) for df in tabelas.values()],
    },
    index=tabelas.keys(),
)
resumo


## 2. Tipos, amostra e estatísticas descritivas

In [ ]:
for nome, df in tabelas.items():
    print("=" * 72)
    print(f"{nome} — dtypes")
    print(df.dtypes)
    print(f"\n{nome} — head")
    display(df.head(3))
    num = df.select_dtypes(include="number")
    if not num.empty:
        print(f"{nome} — describe numérico")
        display(num.describe())


## 3. Nulos e duplicatas

In [ ]:
nulos = []
for nome, df in tabelas.items():
    for col, qtd in df.isna().sum().items():
        if qtd > 0:
            nulos.append(
                {
                    "tabela": nome,
                    "coluna": col,
                    "nulos": int(qtd),
                    "pct": round(100 * qtd / len(df), 3),
                }
            )

nulos_df = pd.DataFrame(nulos)
display(nulos_df if not nulos_df.empty else "Nenhum nulo encontrado.")

dup_resumo = pd.DataFrame(
    {
        "duplicatas_linha_completa": [df.duplicated().sum() for df in tabelas.values()],
    },
    index=tabelas.keys(),
)
dup_resumo


## 4. Chaves únicas e cardinalidade

In [ ]:
chaves = {
    "orders.order_id": orders["order_id"].duplicated().sum(),
    "items (order_id, order_item_id)": items.duplicated(subset=["order_id", "order_item_id"]).sum(),
    "products.product_id": products["product_id"].duplicated().sum(),
    "customers.customer_id": customers["customer_id"].duplicated().sum(),
    "sellers.seller_id": sellers["seller_id"].duplicated().sum(),
}
pd.Series(chaves, name="duplicatas_na_chave")


In [ ]:
relacoes = pd.DataFrame(
    [
        {
            "de": "order_items",
            "para": "orders",
            "chave": "order_id",
            "sem_match": int((~items["order_id"].isin(orders["order_id"])).sum()),
            "cobertura_pct": round(100 * items["order_id"].isin(orders["order_id"]).mean(), 3),
        },
        {
            "de": "orders",
            "para": "customers",
            "chave": "customer_id",
            "sem_match": int((~orders["customer_id"].isin(customers["customer_id"])).sum()),
            "cobertura_pct": round(100 * orders["customer_id"].isin(customers["customer_id"]).mean(), 3),
        },
        {
            "de": "order_items",
            "para": "products",
            "chave": "product_id",
            "sem_match": int((~items["product_id"].isin(products["product_id"])).sum()),
            "cobertura_pct": round(100 * items["product_id"].isin(products["product_id"]).mean(), 3),
        },
        {
            "de": "order_items",
            "para": "sellers",
            "chave": "seller_id",
            "sem_match": int((~items["seller_id"].isin(sellers["seller_id"])).sum()),
            "cobertura_pct": round(100 * items["seller_id"].isin(sellers["seller_id"]).mean(), 3),
        },
    ]
)
relacoes


In [ ]:
# Pedidos sem itens (inconsistência de negócio / fluxo)
pedidos_sem_item = ~orders["order_id"].isin(items["order_id"])
print(f"Pedidos sem itens: {pedidos_sem_item.sum()}")
orders.loc[pedidos_sem_item, "order_status"].value_counts()


## 5. Distribuições e extremos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(items["price"], bins=50, ax=axes[0], color="#3d7ea6")
axes[0].set_title("Distribuição de price (item)")
axes[0].set_xlabel("price (R$)")
sns.histplot(items["freight_value"], bins=50, ax=axes[1], color="#65d6bd")
axes[1].set_title("Distribuição de freight_value (item)")
axes[1].set_xlabel("freight_value (R$)")
plt.tight_layout()
plt.show()

print("price < 0:", (items["price"] < 0).sum())
print("freight < 0:", (items["freight_value"] < 0).sum())
print("freight == 0:", (items["freight_value"] == 0).sum())
print("freight > price:", (items["freight_value"] > items["price"]).sum())
display(items[["price", "freight_value"]].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))
display(items.nlargest(5, "freight_value")[["order_id", "order_item_id", "price", "freight_value"]])


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
status = orders["order_status"].value_counts()
sns.barplot(x=status.index, y=status.values, ax=axes[0], color="#3d7ea6")
axes[0].set_title("Status dos pedidos")
axes[0].tick_params(axis="x", rotation=45)

itens_por_pedido = items.groupby("order_id").size()
sns.histplot(itens_por_pedido, bins=range(1, int(itens_por_pedido.max()) + 2), ax=axes[1], color="#65d6bd")
axes[1].set_title("Itens por pedido")
axes[1].set_xlabel("qtd itens")
plt.tight_layout()
plt.show()

display(status.to_frame("qtd"))
display(itens_por_pedido.describe().to_frame("itens_por_pedido"))


In [ ]:
dims = products[["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]]
display(dims.describe())
print("peso <= 0:", (products["product_weight_g"] <= 0).sum())
print("alguma dimensão <= 0:", ((products["product_length_cm"] <= 0) | (products["product_height_cm"] <= 0) | (products["product_width_cm"] <= 0)).sum())
print("categoria nula:", products["product_category_name"].isna().sum())

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(products["product_weight_g"].dropna(), bins=50, ax=ax, color="#3d7ea6")
ax.set_title("Distribuição de product_weight_g")
ax.set_xlabel("peso (g)")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cli = customers["customer_state"].value_counts().head(10)
vnd = sellers["seller_state"].value_counts().head(10)
sns.barplot(x=cli.index, y=cli.values, ax=axes[0], color="#3d7ea6")
axes[0].set_title("Top 10 UF — clientes")
sns.barplot(x=vnd.index, y=vnd.values, ax=axes[1], color="#65d6bd")
axes[1].set_title("Top 10 UF — vendedores")
plt.tight_layout()
plt.show()


## 6. Contrato da etapa (granularidade e chaves)

| Elemento | Definição |
|----------|-----------|
| Granularidade de item | `order_id` + `order_item_id` |
| Pedido | `order_id` (agregar preço/frete nas perguntas por pedido) |
| Cliente | `orders.customer_id` → `customers` |
| Produto | `order_items.product_id` → `products` |
| Vendedor | `order_items.seller_id` → `sellers` |

Campos indispensáveis às perguntas: `customer_state`, `price`, `freight_value`, `product_weight_g`, dimensões, `order_purchase_timestamp`.

Detalhamento completo: `docs/diagnostico_e_contrato.md`.


## 7. Diagnóstico resumido (sem conclusões de negócio)

1. Cobertura das FKs entre as cinco tabelas é **100%** no sentido item→pedido/produto/vendedor e pedido→cliente.
2. Há **775 pedidos sem itens**, concentrados em status `unavailable`/`canceled` — critério de tratamento fica para a limpeza.
3. Datas estão como texto; nulos de entrega acompanham pedidos não concluídos.
4. Extremos de frete e preço existem; **não excluir automaticamente**.
5. 610 produtos sem categoria e 2 sem peso/dimensões; 4 pesos ≤ 0.
6. Esta etapa **não altera** os CSVs originais e **não antecipa** respostas às três perguntas.
